# BODMAS Data Poisoning Attack Comparison

Simple comparison table for `results/bodmas/100%`.

The notebook ignores the `old` folder and exports the displayed table as a downloadable CSV.

In [1]:
from pathlib import Path
import os

import pandas as pd
from IPython.display import FileLink, display

pd.set_option('display.max_columns', 20)
pd.set_option('display.max_rows', 200)


def find_bodmas_results_dir():
    env_path = os.environ.get('BODMAS_RESULTS_DIR')
    if env_path:
        candidate = Path(env_path).expanduser().resolve()
        if candidate.exists():
            return candidate
        raise FileNotFoundError(f'BODMAS_RESULTS_DIR does not exist: {candidate}')

    start = Path.cwd().resolve()
    for base in (start, *start.parents):
        candidates = [
            base / 'results' / 'bodmas' / '100%',
            base / 'result' / 'bodmas' / '100%',
            base / '..' / 'results' / 'bodmas' / '100%',
            base / '..' / 'result' / 'bodmas' / '100%',
        ]
        for candidate in candidates:
            candidate = candidate.resolve()
            if list(candidate.glob('*/*/*summary_df.csv')):
                return candidate

    raise FileNotFoundError('Could not find results/bodmas/100% or result/bodmas/100%.')


BODMAS_RESULTS_DIR = find_bodmas_results_dir()
print(f'Using BODMAS results directory: {BODMAS_RESULTS_DIR}')

Using BODMAS results directory: /Users/falcon/Machine Learning/Severi Data Poisoning Attack/results/bodmas/100%


In [2]:
REQUESTED_COLUMNS = [
    'sampling_strategy',
    'feat_selector',
    'value_selector',
    'new model - orig_test_set accuracy',
    'new model - mw_test_set - accuracy',
    'evasions - success - percent',
]


def parse_selector_folder(folder_name):
    parts = folder_name.split('__')
    if len(parts) < 5:
        raise ValueError(f'Unexpected selector folder name: {folder_name}')
    return parts[2], parts[3]


def load_summary_row(csv_path):
    rel = csv_path.relative_to(BODMAS_RESULTS_DIR)
    sampling_strategy = rel.parts[0]
    selector_folder = rel.parts[1]
    feat_selector, value_selector = parse_selector_folder(selector_folder)

    df = pd.read_csv(csv_path)
    df = df.loc[:, ~df.columns.str.startswith('Unnamed')]
    if df.empty:
        raise ValueError(f'Empty summary CSV: {csv_path}')

    row = df.iloc[0]
    return {
        'sampling_strategy': sampling_strategy,
        'feat_selector': feat_selector,
        'value_selector': value_selector,
        'new model - orig_test_set accuracy': row['new_model_orig_test_set_accuracy'],
        'new model - mw_test_set - accuracy': row['new_model_mw_test_set_accuracy'],
        'evasions - success - percent': row['evasions_success_percent'],
    }


csv_files = []
for csv_path in sorted(BODMAS_RESULTS_DIR.glob('*/*/*summary_df.csv')):
    rel_parts = csv_path.relative_to(BODMAS_RESULTS_DIR).parts
    if len(rel_parts) != 3:
        continue
    if rel_parts[0] == 'old':
        continue
    csv_files.append(csv_path)

if not csv_files:
    raise FileNotFoundError(f'No current summary CSV files found in {BODMAS_RESULTS_DIR}')

comparison_df = pd.DataFrame([load_summary_row(path) for path in csv_files], columns=REQUESTED_COLUMNS)
comparison_df = comparison_df.sort_values(['sampling_strategy', 'feat_selector', 'value_selector']).reset_index(drop=True)

print(f'Loaded {len(csv_files)} current CSV files. Ignored any files under old/.')
display(comparison_df.style.format({
    'new model - orig_test_set accuracy': '{:.3f}',
    'new model - mw_test_set - accuracy': '{:.3f}',
    'evasions - success - percent': '{:.3f}',
}))

Loaded 24 current CSV files. Ignored any files under old/.


,sampling_strategy,feat_selector,value_selector,new model - orig_test_set accuracy,new model - mw_test_set - accuracy,evasions - success - percent
0,cosine_similarity,combined_shap,combined_shap,99.947,17.249,5.595
1,cosine_similarity,shap_largest_abs,argmin_Nv_sum_abs_shap,99.947,79.354,17.573
2,cosine_similarity,shap_largest_abs,min_population_new,99.956,3.266,6.383
3,distribution_based_distance,combined_shap,combined_shap,99.965,4.956,16.907
4,distribution_based_distance,shap_largest_abs,argmin_Nv_sum_abs_shap,99.974,0.919,96.007
5,distribution_based_distance,shap_largest_abs,min_population_new,99.982,0.061,9.342
6,feature_based_distance,combined_shap,combined_shap,99.930,16.583,5.963
7,feature_based_distance,shap_largest_abs,argmin_Nv_sum_abs_shap,99.939,62.096,34.831
8,feature_based_distance,shap_largest_abs,min_population_new,99.947,3.152,6.252
9,jaccard_distance,combined_shap,combined_shap,99.965,17.669,5.709


In [3]:
output_dir = Path('bodmas_attack_comparison_outputs')
output_dir.mkdir(parents=True, exist_ok=True)
output_csv = output_dir / 'bodmas_100pct_attack_comparison.csv'
comparison_df.to_csv(output_csv, index=False)

print(f'Saved CSV to: {output_csv.resolve()}')
display(FileLink(str(output_csv), result_html_prefix='Download BODMAS comparison CSV: '))

Saved CSV to: /Users/falcon/Machine Learning/Severi Data Poisoning Attack/backdoor_notebook/bodmas_attack_comparison_outputs/bodmas_100pct_attack_comparison.csv


/Users/falcon/Machine Learning/Severi Data Poisoning Attack/backdoor_notebook/bodmas_attack_comparison_outputs/bodmas_100pct_attack_comparison.csv